# Optimized Gradient Method (OGM)

This notebook starts the Lyapunov-function workflow for the optimized gradient method (OGM) on an $L$-smooth convex function $f$. The reference point $x_\star$ satisfies $\nabla f(x_\star)=0$, the initial radius is $\lVert x_0-x_\star\rVert \le R$, and the performance metric is

$$
f(x_N)-f(x_\star).
$$

For a fixed horizon $N$, define $\theta_{-1}=0$,

$$
\theta_N = \frac{1}{2}\left(1+\sqrt{8\theta_{N-1}^2+1}\right),
\qquad
\theta_k = \frac{1}{2}\left(1+\sqrt{4\theta_{k-1}^2+1}\right) \quad (0\le k<N).
$$

The iterates are initialized with $z_0=x_0$ and, for $k=0,\ldots,N-1$,

$$
y_k=x_k-\frac{1}{L}\nabla f(x_k),\qquad
z_{k+1}=z_k-\frac{2\theta_k}{L}\nabla f(x_k),
$$

$$
x_{k+1}=\left(1-\frac{1}{\theta_{k+1}}\right)y_k+\frac{1}{\theta_{k+1}}z_{k+1}.
$$

Block 1 numerical evidence supports the candidate rate

$$
f(x_N)-f(x_\star) \le \frac{L R^2}{2\theta_N^2}.
$$


## Proof Statement


### Theorem

Assume $f$ is convex and $L$-smooth and let $x_{\star}$ satisfy $\nabla f(x_{\star})=0$. Let $\lVert x_{0}-x_{\star}\rVert\le R$. For the optimized gradient method

$$
y_{k}=x_{k}-\frac{1}{L}\nabla f(x_{k}),\qquad
z_{k+1}=z_{k}-\frac{2\theta_{k}}{L}\nabla f(x_{k}),
$$

$$
x_{k+1}=\left(1-\frac{1}{\theta_{k+1}}\right)y_{k}+\frac{1}{\theta_{k+1}}z_{k+1},
$$

with $z_{0}=x_{0}$ and the fixed-horizon OGM coefficients $\theta_{-1}=0$,

$$
\theta_{N}=\frac{1}{2}\left(1+\sqrt{8\theta_{N-1}^{2}+1}\right),
\qquad
\theta_{k}=\frac{1}{2}\left(1+\sqrt{4\theta_{k-1}^{2}+1}\right),\quad 0\le k<N,
$$

the guarantee is

$$
f(x_{N})-f(x_{\star})\le \frac{L R^{2}}{2\theta_{N}^{2}}.
$$

A closed-form Lyapunov certificate for $1\le k<N$ is

$$
V_{k}=\frac{2\theta_{k}^{2}}{\theta_{N}^{2}}\left(f(x_{k})-f(x_{\star})\right)
-\frac{L}{2\theta_{N}^{2}}\lVert x_{0}-x_{\star}\rVert^{2}
-\frac{\theta_{k}^{2}}{L\theta_{N}^{2}}\lVert \nabla f(x_{k})\rVert^{2}
+\frac{L}{2\theta_{N}^{2}}\lVert z_{k+1}-x_{\star}\rVert^{2}.
$$

The terminal boundary expression is

$$
V_{N}=f(x_{N})-f(x_{\star})-\frac{L}{2\theta_{N}^{2}}\lVert x_{0}-x_{\star}\rVert^{2}.
$$


### Proof outline

Let $I(u,v)$ denote the smooth-convex interpolation residual

$$
I(u,v)=f(v)-f(u)+\langle \nabla f(v),u-v\rangle+
\frac{1}{2L}\lVert \nabla f(u)-\nabla f(v)\rVert^{2}\le 0.
$$

The base identity is

$$
V_{1}=\frac{2}{\theta_{N}^{2}}I(x_{\star},x_{0})+
\frac{2}{\theta_{N}^{2}}I(x_{0},x_{1})+
\frac{2\theta_{1}}{\theta_{N}^{2}}I(x_{\star},x_{1}).
$$

For $1\le k<N-1$, using $\theta_{k+1}^{2}=\theta_{k+1}+\theta_{k}^{2}$, the one-step identity is

$$
V_{k+1}-V_{k}=\frac{2\theta_{k}^{2}}{\theta_{N}^{2}}I(x_{k},x_{k+1})+
\frac{2\theta_{k+1}}{\theta_{N}^{2}}I(x_{\star},x_{k+1}).
$$

Finally,

$$
\frac{L}{2\theta_{N}^{2}}\lVert x_{0}-x_{\star}\rVert^{2}-
\left(f(x_{N})-f(x_{\star})\right)+V_{N}=0.
$$

All interpolation residuals are nonpositive, and their multipliers are nonnegative. Therefore the identities imply $V_{N}\le 0$, and the boundary identity gives

$$
f(x_{N})-f(x_{\star})\le \frac{L}{2\theta_{N}^{2}}\lVert x_{0}-x_{\star}\rVert^{2}\le \frac{L R^{2}}{2\theta_{N}^{2}}.
$$


## Imports

In [ ]:
from functools import cache
from pathlib import Path
import sys

_REPO_ROOT = Path.cwd()
while _REPO_ROOT != _REPO_ROOT.parent and not (_REPO_ROOT / "pepflow").exists():
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pepflow as pf  # noqa: E402
import sympy as sp  # noqa: E402

## Function and Parameters

In [ ]:
L = pf.Parameter("L")
R = pf.Parameter("R")
f = pf.SmoothConvexFunction(is_basis=True, tags=["f"], L=L)

## OGM PEP Setup

In [ ]:
@cache
def theta_ogm_value(i: int, N: int) -> float:
    if i == -1:
        return 0.0
    if i == N:
        return 0.5 * (1 + np.sqrt(8 * theta_ogm_value(N - 1, N) ** 2 + 1))
    return 0.5 * (1 + np.sqrt(4 * theta_ogm_value(i - 1, N) ** 2 + 1))


def make_ctx_ogm(ctx_name: str, N, **kwargs) -> pf.PEPContext:
    N_int = int(N)
    ctx = pf.PEPContext(ctx_name).set_as_current()
    theta = [pf.Parameter(f"theta_{i}") for i in range(N_int + 1)]

    x = pf.Vector(is_basis=True, tags=["x_0"])
    z = x
    f.set_stationary_point("x_star")

    for k in range(N_int):
        y = x - (1 / L) * f.grad(x)
        y.add_tag(f"y_{k}")
        z = z - (2 / L) * theta[k] * f.grad(x)
        z.add_tag(f"z_{k + 1}")
        x = (1 - 1 / theta[k + 1]) * y + (1 / theta[k + 1]) * z
        x.add_tag(f"x_{k + 1}")

    return ctx


def get_pep_setup(N, params):
    N_int = int(N)
    params.update({f"theta_{i}": theta_ogm_value(i, N_int) for i in range(N_int + 1)})
    ctx = make_ctx_ogm(f"ctx_{N}", N)
    pb = pf.PEPBuilder(ctx)
    pb.add_initial_constraint(
        ((ctx["x_0"] - ctx["x_star"]) ** 2).le(R**2, name="initial_condition")
    )
    pb.set_performance_metric(f(ctx[f"x_{N}"]) - f(ctx["x_star"]))
    return ctx, pb, f

## Numerical Evidence

In [ ]:
state_path = Path("examples_peppy/ogm/state/ogm_b1.json")
if not state_path.exists():
    state_path = Path("state/ogm_b1.json")

import json  # noqa: E402

state = json.loads(state_path.read_text())
results = state["sweep_results"]

iters = np.array([r["N"] for r in results], dtype=float)
opt_values = np.array([float(r["opt_value"]) for r in results])
rate_values = np.array([1 / (2 * theta_ogm_value(int(N), int(N)) ** 2) for N in iters])

for N, opt, rate in zip(iters.astype(int), opt_values, rate_values):
    print(f"N={N}: PEP={opt:.10f}, L R^2/(2 theta_N^2)={rate:.10f}")

plt.figure(figsize=(6, 4))
plt.plot(iters, rate_values, "r-", label=r"$L R^2/(2	heta_N^2)$")
plt.scatter(iters, opt_values, color="blue", label="PEP values")
plt.xlabel("N")
plt.ylabel(r"$f(x_N)-f(x_\star)$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Block 1 Summary

The recovered numerical values match the candidate fixed-horizon OGM rate $L R^2/(2	heta_N^2)$ up to solver tolerance. Later sections will add dense and relaxed proof solves, closed-form dual variables, partial-sum Lyapunov construction, rank analysis, special-vector identification, coefficient extraction, and symbolic verification.

## Dense Certificate Solve

Block 2 uses the fixed verification horizon `N_verify = 4`. The dense certificate is stored in `state/ogm_dense.json`; the objective value agrees with the Block 1 rate up to solver tolerance.

In [ ]:
import importlib.util
import itertools
import json
from pathlib import Path

state_dir = Path("examples_peppy/ogm/state")
if not state_dir.exists():
    state_dir = Path("state")

b2 = json.loads((state_dir / "ogm_b2.json").read_text())
dense = json.loads((state_dir / "ogm_dense.json").read_text())
relaxed = json.loads((state_dir / "ogm_relaxed.json").read_text())

print("dense opt:", dense["opt_value"])
print("relaxed opt:", relaxed["opt_value"])
print("tau:", relaxed["tau_sol"])
print("basis vectors:")
for v in relaxed["basis_vectors"]:
    print(" ", v)

## Relaxation Pattern

The sparse relaxation keeps only the chain inequalities `f:x_i,x_{i+1}` for `0 <= i < N` and the star-row inequalities `f:x_star,x_j` for `0 <= j <= N`. All other interpolation inequalities are relaxed.

In [ ]:
tol = 1e-7
print(f"dropped constraints: {len(relaxed['relaxed_constraints'])}")
print("active relaxed lambda entries:")
for i, ri in enumerate(relaxed["lambda_row_names"]):
    for j, cj in enumerate(relaxed["lambda_col_names"]):
        value = float(relaxed["lambda_matrix"][i][j])
        if abs(value) > tol:
            print(f"  lambda({ri}, {cj}) = {value:.12f}")

preserved = abs(float(dense["opt_value"]) - float(relaxed["opt_value"])) < 1e-5
print("preserved optimum:", preserved)

## Closed-Form Lambda

Let $\theta_{N}$ denote the fixed-horizon terminal coefficient. The active dual variables are

$$
\lambda_{x_{i},x_{i+1}} = \frac{2\theta_{i}^{2}}{\theta_{N}^{2}}\quad (0\le i<N),
$$

$$
\lambda_{x_{\star},x_{j}}=\frac{2\theta_{j}}{\theta_{N}^{2}}\quad (0\le j<N),
\qquad
\lambda_{x_{\star},x_{N}}=\frac{1}{\theta_{N}}.
$$


In [ ]:
N_int = int(b2["N_verify"])
setup_path = Path("examples_peppy/ogm/ogm_setup.py")
if not setup_path.exists():
    setup_path = Path("ogm_setup.py")

spec = importlib.util.spec_from_file_location("ogm_setup", setup_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Cannot load setup module from {setup_path}")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)


def theta(i, N=N_int):
    return sp.N(mod.theta_ogm_value(i, N), 30)


def idx(tag, N=N_int):
    suffix = tag.split("_", 1)[1]
    return int(suffix) if suffix.isdigit() else N + 1


def lamb(ri, ci, N=N_int):
    i, j = idx(ri, N), idx(ci, N)
    theta_N = theta(N, N)
    if i < N and j == i + 1:
        return 2 * theta(i, N) ** 2 / theta_N**2
    if i == N + 1 and j < N:
        return 2 * theta(j, N) / theta_N**2
    if i == N + 1 and j == N:
        return 1 / theta_N
    return sp.S(0)


row_names = relaxed["lambda_row_names"]
col_names = relaxed["lambda_col_names"]
lambda_candidate = np.array(
    [[float(lamb(ri, cj)) for cj in col_names] for ri in row_names]
)
lambda_solution = np.array(relaxed["lambda_matrix"], dtype=float)
print("lambda max residual:", np.max(np.abs(lambda_candidate - lambda_solution)))
pf.pprint_labeled_matrix(lambda_candidate, row_names, col_names, precision=6)

## Rank-One S Certificate

The Gram dual matrix is rank one up to solver tolerance. With

$$
\tau=\frac{1}{2\theta_{N}^{2}},
$$

use

$$
S = \tau\left(x_{0}-x_{\star} - \frac{2}{L}\sum_{i=0}^{N-1}\theta_{i}\nabla f(x_{i}) - \frac{\theta_{N}}{L}\nabla f(x_{N})\right)^2.
$$


In [ ]:
params_sp = {"L": sp.S(1), "R": sp.S(1)}
ctx, pb, obj = mod.get_pep_setup(sp.S(N_int), params_sp)
pm = pf.ExpressionManager(ctx, resolve_parameters=params_sp)
L_value = sp.S(1)
theta_N = theta(N_int, N_int)
tau = sp.S(1) / (2 * theta_N**2)

S_vec = ctx["x_0"] - ctx["x_star"]
for k in range(N_int):
    S_vec = S_vec - 2 * theta(k, N_int) / L_value * obj.grad(ctx[f"x_{k}"])
S_vec = S_vec - theta_N / L_value * obj.grad(ctx[f"x_{N_int}"])
S_guess = tau * (S_vec**2)

S_candidate = np.array(pm.eval_scalar(S_guess).inner_prod_coords, dtype=float)
S_solution = np.array(relaxed["S_matrix"], dtype=float)
print("S max residual:", np.max(np.abs(S_candidate - S_solution)))
pf.pprint_labeled_matrix(
    S_candidate, relaxed["S_row_names"], relaxed["S_col_names"], precision=6
)

## Fixed-Horizon Proof Identity

The certificate verifies the scalar identity

$$
f(x_{N})-f(x_{\star})-\tau\lVert x_{0}-x_{\star}\rVert^{2}
-\sum_{i,j}\lambda_{ij} I_{ij}+S = 0,
$$

where $I_{ij}$ is the smooth-convex interpolation expression returned by `f.interp_ineq(i, j)`.


In [ ]:
interp_sum = pf.Scalar.zero()
for ri, cj in itertools.product(row_names, col_names):
    coeff = lamb(ri, cj)
    if abs(float(coeff)) > 1e-12:
        interp_sum += coeff * obj.interp_ineq(ri, cj)

x_N = ctx[f"x_{N_int}"]
x_0 = ctx["x_0"]
x_star = ctx["x_star"]
perf = obj(x_N) - obj(x_star)
initial_condition = (x_0 - x_star) ** 2
residual = perf - tau * initial_condition - interp_sum + S_guess
residual_eval = pm.eval_scalar(residual)

inner_residual = np.array(residual_eval.inner_prod_coords, dtype=float)
func_residual = np.array(residual_eval.func_coords, dtype=float)
offset_residual = float(residual_eval.offset)
proof_valid = (
    np.allclose(inner_residual, 0, atol=1e-8)
    and np.allclose(func_residual, 0, atol=1e-8)
    and abs(offset_residual) < 1e-8
)
print("proof valid:", proof_valid)
print("inner max residual:", np.max(np.abs(inner_residual)))
print("function-value max residual:", np.max(np.abs(func_residual)))
print("offset residual:", offset_residual)
pf.pprint_labeled_matrix(
    inner_residual, relaxed["S_row_names"], relaxed["S_col_names"], precision=3
)

## Lyapunov Partial-Sum Construction

Block 3 turns the full-PEP certificate into partial sums. With the active duals from Block 2, the increments are

$$
V_{k+1}-V_k = \lambda_{x_k,x_{k+1}} I_{x_k,x_{k+1}}
+ \lambda_{x_\star,x_{k+1}} I_{x_\star,x_{k+1}},
\qquad 0\le k<N,
$$

with the boundary term $\lambda_{x_\star,x_0} I_{x_\star,x_0}$ added before the loop. Since the full certificate has a single direct rank-one $S$ term, the construction subtracts that $S$ only at the terminal boundary $V_N$. The resulting rank profile is expected to have constant interior rank, with a lower-rank final boundary term.

In [ ]:
b3 = json.loads((state_dir / "ogm_b3.json").read_text())
print("extra duals:", b3["extra_duals"])
print("grouping:")
print("  start with lambda(x_star, x_0)")
print("  step k adds lambda(x_k, x_{k+1}) and lambda(x_star, x_{k+1})")
print("  terminal step subtracts the direct rank-one S")

## Direct S Boundary Term

The same rank-one term from Block 2 is reused at the terminal boundary:

$$
S = \tau\left(x_0-x_\star - \frac{2}{L}\sum_{i=0}^{N-1}\theta_i\nabla f(x_i) - \frac{\theta_N}{L}\nabla f(x_N)\right)^2.
$$

In [ ]:
def terminal_S():
    theta_N = theta(N_int, N_int)
    tau = sp.S(1) / (2 * theta_N**2)
    S_vec = ctx["x_0"] - ctx["x_star"]
    for k in range(N_int):
        S_vec = S_vec - 2 * theta(k, N_int) / L_value * obj.grad(ctx[f"x_{k}"])
    S_vec = S_vec - theta_N / L_value * obj.grad(ctx[f"x_{N_int}"])
    return tau * (S_vec**2)

## Partial Sums and Rank Profile

In [ ]:
lyap = [pf.Scalar.zero()]
partial_sum = pf.Scalar.zero()
partial_sum += lamb("x_star", "x_0") * obj.interp_ineq("x_star", "x_0")

for step in range(N_int):
    partial_sum += lamb(f"x_{step}", f"x_{step + 1}") * obj.interp_ineq(
        f"x_{step}", f"x_{step + 1}"
    )
    partial_sum += lamb("x_star", f"x_{step + 1}") * obj.interp_ineq(
        "x_star", f"x_{step + 1}"
    )
    if step == N_int - 1:
        lyap.append(partial_sum - terminal_S())
    else:
        lyap.append(partial_sum)

rank_tolerance = 1e-4
ranks = []
for k, Vk in enumerate(lyap):
    matrix = pm.eval_scalar(Vk).inner_prod_coords.astype(float)
    rank = int(np.linalg.matrix_rank(matrix, tol=rank_tolerance))
    ranks.append(rank)
    print(f"rank V_{k}: {rank}")
    if k == 0:
        print()

print("Interior rank is constant:", len(set(ranks[1:N_int])) == 1)

## Coverage Check

In [ ]:
M_final = pm.eval_scalar(lyap[N_int]).inner_prod_coords.astype(float)
rank_final = int(np.linalg.matrix_rank(M_final, tol=rank_tolerance))
print(f"lyap[{N_int}] rank:", rank_final)
print("Coverage check: lyap[N] rank should be 0 or 1 (boundary identity term)")
print("Stored rank profile matches:", ranks == b3["rank_profile"])

## Identify the vectors composing the Lyapunov function

Block 4 starts from the Block 3 partial sums and searches for interpretable vectors spanning each low-rank quadratic form. The goal is to replace numerical column-space information with a clean $k$-indexed template for $V_k$.

In [ ]:
b4 = json.loads((state_dir / "ogm_b4.json").read_text())
print("stored rank profile:", b4["rank_profile"])
print("current rank profile:", ranks)
print("rank profile matches:", ranks == b4["rank_profile"])

### Candidate-vector scan

The candidate families include point-to-solution gaps, gradients, auxiliary OGM iterates $z_k$, and simple accumulated-gradient vectors. Duplicate and zero candidates are skipped before the column-space scan.

In [ ]:
def candidate_coord(vec):
    return np.array(pm.eval_vector(vec).coords, dtype=float).ravel()


def add_candidate(candidates, label, vec):
    coords = candidate_coord(vec)
    if np.linalg.norm(coords) < 1e-10:
        return
    for _, _, old_coords in candidates:
        if np.linalg.norm(coords - old_coords) < 1e-9:
            return
    candidates.append((label, vec, coords))


def accumulated_grad(k):
    vec = pf.Vector.zero()
    for i in range(k):
        vec = vec + theta(i, N_int) * obj.grad(ctx[f"x_{i}"])
    return vec


candidates = []
for i in range(N_int + 1):
    add_candidate(candidates, f"x_{i}-x_star", ctx[f"x_{i}"] - ctx["x_star"])
    add_candidate(candidates, f"grad_f(x_{i})", obj.grad(ctx[f"x_{i}"]))
for i in range(1, N_int + 1):
    add_candidate(candidates, f"z_{i}-x_star", ctx[f"z_{i}"] - ctx["x_star"])
for i in range(N_int + 1):
    add_candidate(candidates, f"sum_theta_grad_{i}", accumulated_grad(i))

candidate_vectors = [vec for _, vec, _ in candidates]
candidate_labels = {id(vec): label for label, vec, _ in candidates}
print("candidate count:", len(candidates))

In [ ]:
from pepflow.lyapunov_utils import vectors_in_column_space

for k in range(1, N_int + 1):
    in_col = vectors_in_column_space(
        lyap[k],
        candidate_vectors,
        pep_context=ctx,
        resolve_parameters=params_sp,
        rtol=1e-4,
        atol=1e-4,
    )
    print(f"V_{k} column-space candidates:")
    for vec in in_col:
        print("  ", candidate_labels[id(vec)])

### Selected basis pattern

The sparse-basis scan selects the diagonalizing interior template

$$
\mathcal B_k = \left[x_0-x_\star,\; \nabla f(x_k),\; z_{k+1}-x_\star\right],
\qquad 1\le k<N.
$$

The terminal boundary is rank one with basis $[x_0-x_\star]$.

In [ ]:
def V_k_basis(k):
    if k == 0:
        return []
    if k == N_int:
        return [ctx["x_0"] - ctx["x_star"]]
    return [
        ctx["x_0"] - ctx["x_star"],
        obj.grad(ctx[f"x_{k}"]),
        ctx[f"z_{k + 1}"] - ctx["x_star"],
    ]


def V_k_basis_labels(k):
    if k == 0:
        return []
    if k == N_int:
        return ["x_0-x_star"]
    return ["x_0-x_star", f"grad_f(x_{k})", f"z_{k + 1}-x_star"]


for k in range(1, N_int + 1):
    basis = V_k_basis(k)
    coords = np.column_stack([candidate_coord(v) for v in basis])
    basis_rank = int(np.linalg.matrix_rank(coords, tol=1e-7))
    print(f"k={k}: rank {basis_rank} basis {V_k_basis_labels(k)}")

### Coefficient matrices

For $1\le k<N$, the basis order is $[x_0-x_\star,\nabla f(x_k),z_{k+1}-x_\star]$. With $\tau=1/(2\theta_N^2)$,

$$
C_k = \operatorname{diag}\left(-\tau,\; -\frac{\theta_k^2}{\theta_N^2},\; \tau\right).
$$

For the terminal boundary $k=N$, the coefficient matrix in basis $[x_0-x_\star]$ is $[-\tau]$.

In [ ]:
from pepflow.lyapunov_utils import find_symmetric_coefficient_matrix


def coeff_pattern(k, N=N_int):
    theta_N = theta(N, N)
    tau = sp.S(1) / (2 * theta_N**2)
    if k == 0:
        return []
    if k == N:
        return [[-tau]]
    return [
        [-tau, sp.S(0), sp.S(0)],
        [sp.S(0), -(theta(k, N) ** 2) / theta_N**2, sp.S(0)],
        [sp.S(0), sp.S(0), tau],
    ]


for k in range(1, N_int + 1):
    basis = V_k_basis(k)
    labels = V_k_basis_labels(k)
    C = find_symmetric_coefficient_matrix(
        lyap[k], basis, pep_context=ctx, resolve_parameters=params_sp, span_tol=1e-5
    )
    C_formula = np.array(coeff_pattern(k), dtype=float)
    residual = np.max(np.abs(C - C_formula))
    print(f"k={k}: formula residual {residual:.3e}")
    pf.pprint_labeled_matrix(C, labels, labels, precision=6)

### Block 4 conclusion

For $1\le k<N$, the candidate Lyapunov quadratic part is

$$
V_k = -\tau\lVert x_0-x_\star\rVert^2
-\frac{\theta_k^2}{\theta_N^2}\lVert \nabla f(x_k)\rVert^2
+\tau\lVert z_{k+1}-x_\star\rVert^2
$$

plus the function-value terms already carried by the partial sums. Block 5 will symbolically verify the step, base, and boundary identities for the closed-form expression.

## Symbolic Step Recursion Verification

The identity being verified is

$$
V_{k+1}-V_{k}=\frac{2\theta_{k}^{2}}{\theta_{N}^{2}}I(x_{k},x_{k+1})+
\frac{2\theta_{k+1}}{\theta_{N}^{2}}I(x_{\star},x_{k+1}).
$$

The residual $\mathrm{LHS}-\mathrm{RHS}$ should simplify to zero after using $\theta_{k+1}^{2}=\theta_{k+1}+\theta_{k}^{2}$.

In [ ]:
b5 = json.loads((state_dir / "ogm_b5.json").read_text())


def simplified_residual_parts(evaluated, substitutions=None):
    parts = []
    for raw in [evaluated.inner_prod_coords, evaluated.func_coords, [evaluated.offset]]:
        arr = np.array(raw, dtype=object)
        simplified = []
        for entry in arr.ravel():
            value = sp.factor(sp.simplify(entry))
            if substitutions:
                old_value = None
                while old_value != value:
                    old_value = value
                    value = sp.factor(sp.simplify(value.xreplace(substitutions)))
            simplified.append(value)
        parts.append(np.array(simplified, dtype=object).reshape(arr.shape))
    return parts


def residual_is_zero(parts):
    return all(
        all(entry == 0 for entry in np.array(part, dtype=object).ravel())
        for part in parts
    )


a_sym, b_sym, theta_N_sym = sp.symbols("a b theta_N", positive=True)
tau_sym = sp.Rational(1, 2) / theta_N_sym**2
ctx_step = pf.PEPContext("ogm_step_symbolic").set_as_current()
f_step = pf.SmoothConvexFunction(is_basis=True, tags=["f_step"], L=sp.S(1))
x0_step = pf.Vector(is_basis=True, tags=["x_0"])
xk_step = pf.Vector(is_basis=True, tags=["x_k"])
xk1_step = pf.Vector(is_basis=True, tags=["x_{k+1}"])
xs_step = f_step.set_stationary_point("x_star")
gk_step = f_step.grad(xk_step)
gk1_step = f_step.grad(xk1_step)
zk1_step = b_sym * xk1_step - (b_sym - 1) * (xk_step - gk_step)
zk2_step = zk1_step - 2 * b_sym * gk1_step

V_k_step = (
    (2 * a_sym**2 / theta_N_sym**2) * (f_step(xk_step) - f_step(xs_step))
    - tau_sym * (x0_step - xs_step) ** 2
    - (a_sym**2 / theta_N_sym**2) * (gk_step**2)
    + tau_sym * ((zk1_step - xs_step) ** 2)
)
V_k1_step = (
    (2 * b_sym**2 / theta_N_sym**2) * (f_step(xk1_step) - f_step(xs_step))
    - tau_sym * (x0_step - xs_step) ** 2
    - (b_sym**2 / theta_N_sym**2) * (gk1_step**2)
    + tau_sym * ((zk2_step - xs_step) ** 2)
)
step_rhs = (2 * a_sym**2 / theta_N_sym**2) * f_step.interp_ineq(
    "x_k", "x_{k+1}", sympy_mode=True
) + (2 * b_sym / theta_N_sym**2) * f_step.interp_ineq(
    "x_star", "x_{k+1}", sympy_mode=True
)
step_diff = V_k1_step - V_k_step - step_rhs
pm_step = pf.ExpressionManager(ctx_step, resolve_parameters={})
step_parts = simplified_residual_parts(
    pm_step.eval_scalar(step_diff, sympy_mode=True), {b_sym**2: b_sym + a_sym**2}
)
print("Step identity zero:", residual_is_zero(step_parts))
pf.pprint_labeled_matrix(
    step_parts[0], ctx_step.basis_vectors_math_exprs(), precision=3
)
print("Function residual zero:", all(v == 0 for v in step_parts[1].ravel()))

## Base Case Symbolic Verification

The identity being verified is

$$
V_{1}=\frac{2}{\theta_{N}^{2}}I(x_{\star},x_{0})+
\frac{2}{\theta_{N}^{2}}I(x_{0},x_{1})+
\frac{2\theta_{1}}{\theta_{N}^{2}}I(x_{\star},x_{1}).
$$

The residual $\mathrm{LHS}-\mathrm{RHS}$ should simplify to zero after using $\theta_{1}^{2}=\theta_{1}+1$.

In [ ]:
theta_1_sym = sp.symbols("theta_1", positive=True)
ctx_base = pf.PEPContext("ogm_base_symbolic").set_as_current()
f_base = pf.SmoothConvexFunction(is_basis=True, tags=["f_base"], L=sp.S(1))
x0_base = pf.Vector(is_basis=True, tags=["x_0"])
xs_base = f_base.set_stationary_point("x_star")
g0_base = f_base.grad(x0_base)
z1_base = x0_base - 2 * g0_base
x1_base = (1 - sp.S(1) / theta_1_sym) * (x0_base - g0_base) + (
    sp.S(1) / theta_1_sym
) * z1_base
x1_base.add_tag("x_1")
g1_base = f_base.grad(x1_base)
z2_base = z1_base - 2 * theta_1_sym * g1_base
V1_base = (
    (2 * theta_1_sym**2 / theta_N_sym**2) * (f_base(x1_base) - f_base(xs_base))
    - tau_sym * (x0_base - xs_base) ** 2
    - (theta_1_sym**2 / theta_N_sym**2) * (g1_base**2)
    + tau_sym * ((z2_base - xs_base) ** 2)
)
base_rhs = (
    (2 / theta_N_sym**2) * f_base.interp_ineq("x_star", "x_0", sympy_mode=True)
    + (2 / theta_N_sym**2) * f_base.interp_ineq("x_0", "x_1", sympy_mode=True)
    + (2 * theta_1_sym / theta_N_sym**2)
    * f_base.interp_ineq("x_star", "x_1", sympy_mode=True)
)
base_diff = V1_base - base_rhs
pm_base = pf.ExpressionManager(ctx_base, resolve_parameters={})
base_parts = simplified_residual_parts(
    pm_base.eval_scalar(base_diff, sympy_mode=True), {theta_1_sym**2: theta_1_sym + 1}
)
print("Base identity zero:", residual_is_zero(base_parts))
pf.pprint_labeled_matrix(
    base_parts[0], ctx_base.basis_vectors_math_exprs(), precision=3
)
print("Function residual zero:", all(v == 0 for v in base_parts[1].ravel()))

## Base Case and Boundary Symbolic Verification

The base case has been verified above. The terminal boundary identity is verified separately below.

### Boundary Identity Symbolic Verification

The identity being verified is

$$
\frac{1}{2\theta_{N}^{2}}\lVert x_{0}-x_{\star}\rVert^{2}-\left(f(x_{N})-f(x_{\star})\right)+V_{N}=0,
$$

where $V_{N}=f(x_{N})-f(x_{\star})-\frac{1}{2\theta_{N}^{2}}\lVert x_{0}-x_{\star}\rVert^{2}$. The residual $\mathrm{LHS}-\mathrm{RHS}$ should simplify to zero.

In [ ]:
ctx_boundary = pf.PEPContext("ogm_boundary_symbolic").set_as_current()
f_boundary = pf.SmoothConvexFunction(is_basis=True, tags=["f_boundary"], L=sp.S(1))
x0_boundary = pf.Vector(is_basis=True, tags=["x_0"])
xn_boundary = pf.Vector(is_basis=True, tags=["x_N"])
xs_boundary = f_boundary.set_stationary_point("x_star")
Vn_boundary = (
    f_boundary(xn_boundary)
    - f_boundary(xs_boundary)
    - tau_sym * (x0_boundary - xs_boundary) ** 2
)
boundary_diff = (
    tau_sym * (x0_boundary - xs_boundary) ** 2
    - (f_boundary(xn_boundary) - f_boundary(xs_boundary))
    + Vn_boundary
)
pm_boundary = pf.ExpressionManager(ctx_boundary, resolve_parameters={})
boundary_parts = simplified_residual_parts(
    pm_boundary.eval_scalar(boundary_diff, sympy_mode=True)
)
print("Boundary identity zero:", residual_is_zero(boundary_parts))
pf.pprint_labeled_matrix(
    boundary_parts[0], ctx_boundary.basis_vectors_math_exprs(), precision=3
)
print("Function residual zero:", all(v == 0 for v in boundary_parts[1].ravel()))

## Later Lyapunov Blocks

The partial-sum Lyapunov sequence now has constant interior rank. Later blocks will identify special spanning vectors, extract closed-form coefficients, and verify symbolic step, base, and boundary identities.